In [ ]:
pip install ultralytics

In [ ]:
import os
import yaml
import shutil
import random
import kagglehub
from ultralytics import YOLO
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
downloaded_data_dir = kagglehub.dataset_download("andrewmvd/car-plate-detection")
print(f"Downloaded Kaggle dataset to {downloaded_data_dir}")

downloaded_images_dir = os.path.join(downloaded_data_dir, "images")
downloaded_annotations_dir = os.path.join(downloaded_data_dir, "annotations")

working_dir = "/kaggle/working"

images_dir = os.path.join(working_dir, "images")
train_images_dir = os.path.join(images_dir, "train")
val_images_dir = os.path.join(images_dir, "val")

labels_dir = os.path.join(working_dir, "labels")
train_labels_dir = os.path.join(labels_dir, "train")
val_labels_dir = os.path.join(labels_dir, "val")


In [ ]:
def convert_min_max_to_center_dims(width, height, xmin, xmax, ymin, ymax):
    x_center = ((xmin + xmax) / 2) / width
    y_center = ((ymin + ymax) / 2) / height
    w = (xmax - xmin) / width
    h = (ymax - ymin) / height
    return x_center, y_center, w, h

In [ ]:
def prepare_yolo_dataset(train_ratio=0.75):
    if os.path.exists(working_dir):
        shutil.rmtree(working_dir)

    os.makedirs(train_images_dir)
    os.makedirs(val_images_dir)
    os.makedirs(train_labels_dir)
    os.makedirs(val_labels_dir)

    # Shuffle the image files
    image_files = [f for f in os.listdir(downloaded_images_dir) if f.endswith((".png",".jpg",".jpeg"))]
    random.shuffle(image_files)

    # Split the image files for training and validation
    split_index = int(len(image_files) * train_ratio)
    train_files = image_files[:split_index]
    val_files = image_files[split_index:]

    # -------- TRAIN --------
    for img in train_files:
        base = os.path.splitext(img)[0]
        xml_file = base + ".xml"

        original_image_path = os.path.join(downloaded_images_dir, img)
        train_image_path = os.path.join(train_images_dir, img)
        shutil.copy(original_image_path, train_image_path)

        original_xml_path = os.path.join(downloaded_annotations_dir, xml_file)
        label_path = os.path.join(train_labels_dir, base + ".txt")

        lines = []

        if os.path.exists(original_xml_path):
            tree = ET.parse(original_xml_path)
            root = tree.getroot()

            width = float(root.find("size/width").text)
            height = float(root.find("size/height").text)

            for obj in root.findall("object"):
                xmin = float(obj.find("bndbox/xmin").text)
                xmax = float(obj.find("bndbox/xmax").text)
                ymin = float(obj.find("bndbox/ymin").text)
                ymax = float(obj.find("bndbox/ymax").text)

                x_center, y_center, w, h = convert_min_max_to_center_dims(width, height, xmin, xmax, ymin, ymax)
                lines.append(f"0 {x_center} {y_center} {w} {h}")

        with open(label_path, "w") as f:
            f.write("\n".join(lines))

    # -------- VALIDATION --------
    for img in val_files:
        base = os.path.splitext(img)[0]
        xml_file = base + ".xml"

        original_image_path = os.path.join(downloaded_images_dir, img)
        val_image_path = os.path.join(val_images_dir, img)
        shutil.copy(original_image_path, val_image_path)

        original_xml_path = os.path.join(downloaded_annotations_dir, xml_file)
        label_path = os.path.join(val_labels_dir, base + ".txt")

        lines = []

        if os.path.exists(original_xml_path):
            tree = ET.parse(original_xml_path)
            root = tree.getroot()

            width = float(root.find("size/width").text)
            height = float(root.find("size/height").text)

            for obj in root.findall("object"):
                xmin = float(obj.find("bndbox/xmin").text)
                xmax = float(obj.find("bndbox/xmax").text)
                ymin = float(obj.find("bndbox/ymin").text)
                ymax = float(obj.find("bndbox/ymax").text)

                x_center, y_center, w, h = convert_min_max_to_center_dims(width, height, xmin, xmax, ymin, ymax)
                lines.append(f"0 {x_center} {y_center} {w} {h}")

        with open(label_path, "w") as f:
            f.write("\n".join(lines))

    print("Dataset prepared successfully.")
    print("Train images:", len(train_files))
    print("Validation images:", len(val_files))

In [ ]:
def create_yolo_data_yaml(working_path):
    data = {
        "path": working_dir,
        "train": "images/train",
        "val": "images/val",
        "names": {
            0: "license"
        }
    }

    yaml_path = os.path.join(working_path, "data.yaml")

    with open(yaml_path, "w") as f:
        yaml.dump(data, f, sort_keys=False)

    print(f"YOLO config file created at: {yaml_path}")
    return yaml_path

In [ ]:
prepare_yolo_dataset()

In [ ]:
create_yolo_data_yaml(working_dir)

In [ ]:
model = YOLO('yolov8s.yaml')
config_path = "/kaggle/working/data.yaml"

In [ ]:
model.train(data=config_path, epochs=400, batch=32, optimizer='Adam', lr0=0.001, patience=20)

In [ ]:
results = pd.read_csv("runs/detect/train4/results.csv")

plt.plot(results["epoch"], results["val/box_loss"], label='val')
plt.plot(results["epoch"], results["train/box_loss"], label='train')
plt.xlabel("Epoch")
plt.ylabel("box_loss")
plt.legend()
plt.title("YOLOv8 Training Performance")
plt.show()

In [ ]:
# new_model = YOLO('/content/runs/detect/train/weights/last.pt')
# new_model.train(data=config_path, epochs=400, batch=32, optimizer='Adam', lr0=0.001, patience=20)